In [5]:
import numpy as np
import random

# --- Environment ---
class GridWorld:
    def __init__(self):
        self.grid = np.array([
            ['S', '.', '.', '.'],
            ['.', 'X', '.', '.'],
            ['.', '.', '.', 'G']
        ])
        self.start = (0, 0)
        self.goal = (2, 3)
        self.pos = self.start
        self.actions = ['up', 'down', 'left', 'right']
    
    def reset(self):
        self.pos = self.start
        return self.pos
    
    def step(self, action):
        i, j = self.pos
        # Move according to action
        if action == 'up': i -= 1
        elif action == 'down': i += 1
        elif action == 'left': j -= 1
        elif action == 'right': j += 1
        
        # Boundary / obstacle checks
        if i < 0 or i >= 3 or j < 0 or j >= 4 or self.grid[i][j] == 'X':
            reward = -1  # invalid move
            next_state = self.pos
        else:
            self.pos = (i, j)
            if self.pos == self.goal:
                reward = 1  # goal reward
            else:
                reward = -0.1  # small step penalty
            next_state = self.pos
        
        done = self.pos == self.goal
        return next_state, reward, done


# --- Agent ---
class QLearningAgent:
    def __init__(self, env, alpha=0.1, gamma=0.9, epsilon=0.5):
        self.q_table = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.env = env
    
    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)
    
    def choose_action(self, state):
        # epsilon-greedy
        if random.random() < self.epsilon:
            return random.choice(self.env.actions)
        else:
            q_values = [self.get_q(state, a) for a in self.env.actions]
            max_q = max(q_values)
            best_actions = [a for a, q in zip(self.env.actions, q_values) if q == max_q]
            return random.choice(best_actions)
    
    def learn(self, state, action, reward, next_state, done):
        old_q = self.get_q(state, action)
        next_qs = [self.get_q(next_state, a) for a in self.env.actions]
        max_next_q = max(next_qs) if not done else 0
        new_q = old_q + self.alpha * (reward + self.gamma * max_next_q - old_q)
        self.q_table[(state, action)] = new_q


# --- Training ---
env = GridWorld()
agent = QLearningAgent(env)

num_episodes = 500
max_steps = 20

for episode in range(num_episodes):
    state = env.reset()
    done = False
    for step in range(max_steps):
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.learn(state, action, reward, next_state, done)
        state = next_state
        if done:
            break
    
    # decay epsilon
    agent.epsilon = max(0.05, agent.epsilon * 0.99)

print("\n✅ Training complete!\n")


# --- Policy visualization ---
arrow_map = {'up':'↑', 'down':'↓', 'left':'←', 'right':'→'}
policy_grid = np.full_like(env.grid, ' ')

for i in range(env.grid.shape[0]):
    for j in range(env.grid.shape[1]):
        state = (i, j)
        if env.grid[i][j] in ['X', 'G', 'S']:
            policy_grid[i][j] = env.grid[i][j]
        else:
            q_values = [agent.get_q(state, a) for a in env.actions]
            best_action = env.actions[np.argmax(q_values)]
            policy_grid[i][j] = arrow_map[best_action]

print("🗺️ Learned Policy Map:")
for row in policy_grid:
    print(' '.join(row))



✅ Training complete!

🗺️ Learned Policy Map:
S → ↓ ↓
↓ X ↓ ↓
→ → → G



--- Final Policy ---
D L D D 
U  X  U U 
U L U  G  
